In [13]:
# imports
from judge_utils import load_all_results, main_path, results_path
import json
import os
import pandas as pd

In [14]:
# Load results into dataframe
df_main = load_all_results(results_path)

In [15]:
# Reorder
df_main = df_main.sort_values('run_time').reset_index(drop=True)

In [16]:
# Merge columns for easier lookup
df_main["merge_key"] = df_main.apply(
    lambda row: (
        row["structure_id"],
        row["use_img"],
        row["use_json"],
        row["shot"]
    ),
    axis=1
)

judge_files = {
    "BASE": os.path.join(main_path, "analysis", "judge_analysis_BASE.json"),
    "CLAR_Q": os.path.join(main_path, "analysis", "judge_analysis_CLAR_Q.json"),
    "COMM_SH_REF": os.path.join(main_path, "analysis", "judge_analysis_COMM_SH_REF.json"),
    "IMPL_REF": os.path.join(main_path, "analysis", "judge_analysis_IMPL_REF.json")
}

for label, filepath in judge_files.items():
    try:
        with open(filepath, "r") as f:
            judge_data = json.load(f)
            
        rating_dict = {}
        for entry in judge_data:
            key = (
                entry["structure_id"],
                entry["use_img"],
                entry["use_json"],
                entry["shot"]
            )
            rating_dict[key] = entry["rating"]

        # Map the new column from the dictionary, using the merged key
        df_main[label] = df_main["merge_key"].map(rating_dict)
    except Exception:
        #print(f"{filepath} missing")
        continue  # or handle error

In [17]:
# Load the metrics from "parsed_actions_with_metrics2.json (version with custom edit distance metric)"
with open(os.path.join(main_path, "analysis", "parsed_actions_with_metrics_fixed.json"), "r") as f:
    metrics_data = json.load(f)

In [18]:
# Build a dictionary keyed by the composite key from metrics_data
metrics_dict = {}
for entry in metrics_data:
    key = (
        entry["structure_id"],
        entry["use_img"],
        entry["use_json"],
        entry["shot"]
    )
    metrics_dict[key] = {
        "similarity": entry["similarity"]
    }

# Map each metric onto df_main using the composite key column
df_main["similarity"] = df_main["merge_key"].map(
    lambda key: metrics_dict[key]["similarity"] if key in metrics_dict else None
)

# Optionally drop the temporary composite key column if no longer needed:
df_main.drop(columns=["merge_key"], inplace=True)

In [19]:
df_main = df_main.drop(columns=["json_file", "Model", "Quantization", "Device", "Number of models",	"Max new tokens",	"Repetition Penalty",	"Max rounds", "json_file"])

In [20]:
df_main.value_counts()


run_time            structure_id         use_img  use_json  num_rounds  total_time_min  finished_by_architect  shot       BASE       similarity
2025-02-11-1829-56  C1_bell              True     False     5           1.0             True                   one-shot   3           0.04         1
2025-02-12-1420-09  C14_diagonal-zigzag  False    True      50          10.0            False                  one-shot   1           0.00         1
2025-02-12-1845-27  C16_bloody-sword     False    True      50          9.0             False                  one-shot   1           0.00         1
2025-02-12-1704-55  C16_bloody-sword     True     True      33          99.0            False                  zero-shot  Undefined  -0.05         1
2025-02-12-1703-39  C16_bloody-sword     True     True      50          1.0             False                  one-shot   1           0.00         1
                                                                                                               

In [22]:
df_main[df_main["BASE"] == 3].value_counts()

run_time            structure_id      use_img  use_json  num_rounds  total_time_min  finished_by_architect  shot       BASE  similarity
2025-02-11-1829-56  C1_bell           True     False     5           1.0             True                   one-shot   3     -0.02         1
2025-02-11-2041-13  C2_black-hole     True     True      8           1.0             True                   zero-shot  3     -0.21         1
2025-02-12-0818-44  C9_asterisk       True     True      3           1.0             True                   zero-shot  3      0.47         1
2025-02-12-1702-41  C16_bloody-sword  True     False     5           1.0             True                   zero-shot  3     -0.22         1
dtype: int64

In [23]:
df_main[(df_main["BASE"] == 1) & (df_main["similarity"] > 0)].value_counts()

run_time            structure_id                use_img  use_json  num_rounds  total_time_min  finished_by_architect  shot       BASE  similarity  mean_similarity
2025-02-11-1830-50  C1_bell                     True     False     15          90.0            False                  zero-shot  1     0.45        0.503136           1
2025-02-12-1437-37  C15_double_stairs           True     False     50          8.0             False                  zero-shot  1     0.52        0.503136           1
2025-02-12-0928-55  C11_broken_heart            False    True      50          9.0             False                  one-shot   1     0.50        0.503136           1
2025-02-12-1122-25  C12_diagonal-Ls             True     True      26          4.0             True                   zero-shot  1     0.28        0.503136           1
2025-02-12-1126-19  C12_diagonal-Ls             False    True      50          9.0             False                  one-shot   1     0.50        0.503136          

In [21]:
df_main[df_main["similarity"] > 0].value_counts()
# Compute the mean similarity score (ignores NaNs by default)
mean_similarity = df_main["similarity"].mean()

# Add a new column with this value repeated for all rows
df_main["mean_similarity"] = mean_similarity


In [22]:
df_main[df_main["similarity"] > 0].value_counts()

run_time            structure_id                use_img  use_json  num_rounds  total_time_min  finished_by_architect  shot       BASE       similarity  mean_similarity
2025-02-11-1829-56  C1_bell                     True     False     5           1.0             True                   one-shot   3          0.04        0.092458           1
2025-02-11-2014-53  C1_bell                     True     True      11          1.0             True                   zero-shot  1          0.90        0.092458           1
2025-02-12-1136-08  C12_diagonal-Ls             False    True      50          8.0             False                  zero-shot  1          0.29        0.092458           1
2025-02-12-1144-56  C13_eye                     True     False     17          103.0           False                  one-shot   Undefined  0.12        0.092458           1
2025-02-12-1339-55  C13_eye                     True     True      6           1.0             True                   zero-shot  1          

In [ ]:
# Number of rounds analysis when the architect finishes the conversation
df_main[df_main["finished_by_architect"] == True].num_rounds.describe()

In [ ]:
df_main[(df_main["accuracy"] > 0) & (df_main["precision"] > 0)].head()

In [ ]:
# Just 1 over 6 good accuracy ratings the one-shot was used
df_main[df_main["shot"] == "one-shot"].accuracy.value_counts()

In [ ]:
# Convert columns to numeric, coercing errors to NaN if necessary.
df_main["BASE"] = pd.to_numeric(df_main["BASE"], errors="coerce")
df_main["accuracy"] = pd.to_numeric(df_main["accuracy"], errors="coerce")
df_main["precision"] = pd.to_numeric(df_main["precision"], errors="coerce")
df_main["iou"] = pd.to_numeric(df_main["iou"], errors="coerce")

# Then group and compute the meanprecision and iou. Adapt it so that it 
table = df_main.groupby(["shot", "use_img", "use_json"])[["BASE", "accuracy", "precision"]].mean().reset_index()

In [ ]:
latex_table = table.to_latex(float_format="%.2f")